In [ ]:
%%capture
%pip install langchain==0.2.11
%pip install langchain-community==0.2.10
%pip install sentence-transformers
%pip install jq
%pip install chromadb==0.4.24
%pip install --upgrade chromadb
%pip install --force-reinstall numpy==1.26.4

In [ ]:
## Start the Ollama Server

!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start the server in the background
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a few seconds to initialize
time.sleep(5)
print("Ollama server is running in the background!")
!ollama pull llama3

In [ ]:

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import JSONLoader
import json
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
import os
import numpy as np
import logging
import sys
import warnings

## Dynamically restore the expected alias
if not hasattr(np, 'long'):
    np.long = np.int64
if not hasattr(np, 'ulong'):
    np.ulong = np.uint64
from sentence_transformers import CrossEncoder
from google.colab import drive
drive.mount('/content/drive')


## Section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
warnings.warn = warn
warnings.filterwarnings('ignore')

## Write data to output file
def append_to_json(file_path, new_data):
    # 1. Check if file exists and is not empty
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        with open(file_path, 'r', encoding='utf-8') as file:
            # Load existing data into a list
            data_list = json.load(file)
    else:
        # If file doesn't exist, start with an empty list
        data_list = []

    # 2. Append your new labeled data to the list
    data_list.append(new_data)

    # 3. Write everything back to the file with formatting
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data_list, file, indent=4)

## Document loader
def document_loader(file):
    if file is None:
        logging.info( "Please upload a valid JSON file.")
    loader = JSONLoader(
        file_path=file,
        jq_schema='.[]', # Adjust the jq_schema based on your JSON structure
        text_content=False
    #    strict=False
    )
    loaded_document = loader.load()
    return loaded_document

## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=20,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

## Token Embedding
def hf_embedding():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    model_kwargs = {'device': 'cpu'}
    encode_kwargs = {'normalize_embeddings': False}

    embedding_model = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embedding_model


## Initialize vector db
def vector_database(chunks):
    embedding_model = hf_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb


# 1. Load your input JSON file
# Assumes the JSON file is a list of objects or a dict containing questions
input_file_path = "/content/drive/MyDrive/qa_data_test_Final_50.json"
with open(input_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Extract the list of questions (adjust the key based on your JSON structure)
# Example JSON format: [{"question": "What is X?"}, {"question": "How to do Y?"}]
questions = [item["question"] for item in data]
contexts = [item["knowledge"] for item in data]
ground_truths = [item["right_answer"] for item in data]
template = """You are an expert fact-extraction system. You must output EXACTLY the correct answer from the provided context. Do NOT use conversational filler, do NOT add introductory text like 'According to the context', and do NOT output full sentences. Only output the exact answer.
Context: {context}
Question: {question}
Answer:"""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

llm = Ollama(model="llama3",
    temperature=1.0,
    num_predict=200, # Max output tokens
    num_ctx=4096)

## Store input tokens in vector db
splits = document_loader(input_file_path)
chunks = text_splitter(splits)
vectordb = vector_database(chunks)

## define three labels of NLI (Natural Language Inference) cross-encoder
label_mapping = ['contradiction', 'entailment', 'neutral']

## Initialize Retriever chain from langchain
## chain_type_kwargs need to be uncommented for 'PROMPT ON'
## chain_type_kwargs need to be commented for 'PROMPT OFF'
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectordb.as_retriever(),
    return_source_documents=True
   # chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

## Initialize CrossEncoder
nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

## Loop through all objets in input JSON file
for i, (question, context, ground_truth) in enumerate(zip(questions, contexts, ground_truths)):
    try:
        response = qa.invoke(question)
        if isinstance(response, dict):
            answer = response.get("result", "").strip()
        else:
            answer = str(response).strip()

        ## Get the NLI label by comparing ground truth and the answer from llm.
        gt_scores = nli_model.predict([(ground_truth, answer)])
        predicted_label = label_mapping[np.argmax(gt_scores[0])]

        ## Store data in output file
        output_data = {
            "Temperature": 1.0,
            "Context_length": 4096,
            "Context": context,
            "Question":question,
            "Ground_truth": ground_truth,
            "Answer_by_llm": answer,
            "Predicted_label": predicted_label,
            "Chunk_Size": 200,
            "Prompt": "OFF"
        }

        append_to_json('/content/drive/MyDrive/output_test1.json', output_data)

        # Clear memory every 10 iterations
        if i%10==0 and i!=0:
          process.terminate()  # Sends SIGTERM
          try:
              process.wait(timeout=5)  # Wait up to 5 seconds for it to close
              print("Ollama server stopped successfully.")
          except subprocess.TimeoutExpired:
              process.kill()  # Force kill if it hangs
              print("Ollama server force killed.")

          # 4. Restart the server
          process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
          print("Ollama server restarted.")
          !ollama pull llama3
          print("Ollama pull done.")

    except Exception as e:
        logging.error(f"Error during chain invocation: {e}")

from google.colab import drive
drive.mount('/content/drive')

